# Anime Recommendation System - Collaborative Filtering DAE

This notebook builds a high-level recommendation system using a **Multi-Task Denoising Autoencoder (DAE)**. It learns from implicit (presence) and explicit (rating) feedback.

**Dataset**: [Anime Dataset (Jan 1917 to Oct 2025)](https://www.kaggle.com/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025)
**Framework**: JAX + Flax

Based on learnings from the Sprout repository, this model features:
- Dual-head decoder (Presence prediction + Rating regression)
- Uncertainty weighting for multi-task loss balance
- Hybrid absolute/relative rating normalization

## 1. Data Loading and Preprocessing
We will load `ratings.csv` and `details.csv` from the Kaggle dataset. We filter out rare items to create a dense corpus of the top N most popular anime, and filter out users with too few ratings.

In [7]:
import os
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
from flax.training import train_state
import optax
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# config
CORPUS_SIZE = 6000
MIN_USER_RATINGS = 10
BATCH_SIZE = 512
HIDDEN_DIM = 2048
BOTTLENECK_DIM = 512
LEARNING_RATE = 3e-4
DROPOUT_RATE = 0.4
EPOCHS = 10

DATA_DIR = '/kaggle/input/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025/datasets'
RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.csv')
DETAILS_PATH = os.path.join(DATA_DIR, 'details.csv')

print("Loading datasets...")

ratings_df = pd.read_csv(RATINGS_PATH, usecols=['username', 'anime_id', 'score', 'status'])

# Determine top anime for the corpus
anime_counts = ratings_df['anime_id'].value_counts()
corpus_ids = anime_counts.head(CORPUS_SIZE).index.values
corpus_id_to_idx = {aid: idx for idx, aid in enumerate(corpus_ids)}

# Filter ratings to only include corpus items
ratings_df = ratings_df[ratings_df['anime_id'].isin(corpus_id_to_idx)]

# Filter users with less than MIN_USER_RATINGS
user_counts = ratings_df['username'].value_counts()
valid_users = user_counts[user_counts >= MIN_USER_RATINGS].index
ratings_df = ratings_df[ratings_df['username'].isin(valid_users)]

print(f"Retained {len(valid_users)} users and {len(ratings_df)} ratings.")

# Save the corpus mapping for backend use
import json
with open('/kaggle/working/corpus_mapping.json', 'w') as f:
    json.dump({'corpus_ids': corpus_ids.tolist()}, f)
print("Saved corpus mapping to /kaggle/working/corpus_mapping.json")

Loading datasets...
Retained 293194 users and 114845875 ratings.
Saved corpus mapping to /kaggle/working/corpus_mapping.json


## 2. Rating Normalization
We use a hybrid approach that mixes Z-Score normalization (relative user preference) and Absolute Scale normalization. Dropped unrated items act as a strong negative signal.

In [8]:
print("Starting vectorized rating normalization...")

steps = [
    "Mapping anime_ids",
    "Calculating user stats",
    "Processing unrated & dropped",
    "Calculating final normalizations",
    "Constructing Sparse Matrices"
]

for step in tqdm(steps, desc="Preprocessing Pipeline"):
    if step == "Mapping anime_ids":
        ratings_df['idx'] = ratings_df['anime_id'].map(corpus_id_to_idx)
    elif step == "Calculating user stats":
        user_stats = ratings_df[ratings_df['score'] > 0].groupby('username')['score'].agg(['mean', 'std']).rename(columns={'mean': 'user_mean', 'std': 'user_std'})
        ratings_df = ratings_df.join(user_stats, on='username')
        ratings_df['user_mean'] = ratings_df['user_mean'].fillna(5.0)
        ratings_df['user_std'] = ratings_df['user_std'].fillna(2.0)
    elif step == "Processing unrated & dropped":
        ratings_df['processed_score'] = ratings_df['score'].astype(np.float32)
        mask_dropped_unrated = (ratings_df['status'] == 'dropped') & (ratings_df['score'] == 0)
        ratings_df.loc[mask_dropped_unrated, 'processed_score'] = -2.0
        
        mask_watched_unrated = (ratings_df['score'] == 0) & (ratings_df['status'] != 'dropped')
        ratings_df.loc[mask_watched_unrated, 'processed_score'] = ratings_df.loc[mask_watched_unrated, 'user_mean']
        
        ratings_df.loc[mask_dropped_unrated, 'processed_score'] = ratings_df.loc[mask_dropped_unrated, 'user_mean'] - 1.5 * ratings_df.loc[mask_dropped_unrated, 'user_std']
    elif step == "Calculating final normalizations":
        mu = ratings_df['user_mean']
        sigma = ratings_df['user_std'] + 1e-6
        z_score = np.clip((ratings_df['processed_score'] - mu) / sigma, -3.0, 3.0)
        abs_score = np.clip((ratings_df['processed_score'] - 5.5) / 2.5, -2.5, 2.0)
        alpha = np.clip(sigma / 2.6, 0.3, 0.8)
        ratings_df['norm_score'] = np.clip(alpha * z_score + (1.0 - alpha) * abs_score, -2.5, 2.5)
    elif step == "Constructing Sparse Matrices":
        ratings_df['user_idx'] = ratings_df['username'].astype('category').cat.codes
        num_users = ratings_df['user_idx'].nunique()
        
        user_idxs = ratings_df['user_idx'].values
        item_idxs = ratings_df['idx'].values
        norm_scores = ratings_df['norm_score'].values
        is_rated = (ratings_df['score'] > 0).values.astype(np.float32)
        
        col_presence = item_idxs
        data_presence = np.ones_like(item_idxs, dtype=np.float32)
        
        col_ratings = item_idxs + CORPUS_SIZE
        data_ratings = norm_scores
        
        cols = np.concatenate([col_presence, col_ratings])
        rows = np.concatenate([user_idxs, user_idxs])
        data = np.concatenate([data_presence, data_ratings])
        
        X_train_sparse = csr_matrix((data, (rows, cols)), shape=(num_users, CORPUS_SIZE * 2), dtype=np.float32)
        M_train_sparse = csr_matrix((is_rated, (user_idxs, item_idxs)), shape=(num_users, CORPUS_SIZE), dtype=np.float32)

print(f"Completed! Training data shape: {X_train_sparse.shape}")

Starting vectorized rating normalization...


Preprocessing Pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Completed! Training data shape: (293194, 12000)


## 3. Model Architecture (JAX / Flax)
We define a Multi-Task Denoising Autoencoder. The encoder compresses the user profile into a bottleneck, and the dual-head decoder predicts presence (implicit feedback) and ratings (explicit feedback).

In [9]:
class Recommender(nn.Module):
    hidden_dim: int = HIDDEN_DIM
    bottleneck_dim: int = BOTTLENECK_DIM
    corpus_size: int = CORPUS_SIZE

    @nn.compact
    def __call__(self, x, training: bool = False):
        # --- Encoder ---
        h = nn.Dense(self.hidden_dim)(x)
        h = nn.swish(h)
        bottleneck = nn.Dense(self.bottleneck_dim, name="bottleneck")(h)
        
        if training:
            # Add noise to latent space for regularization
            rng = self.make_rng('noise')
            noise = random.normal(rng, bottleneck.shape) * 0.1
            z = bottleneck + noise
        else:
            z = bottleneck
            
        # --- Decoder Head 1: Item Presence (Logits) ---
        d1 = nn.Dense(self.hidden_dim // 2)(z)
        d1 = nn.swish(d1)
        d1 = nn.Dense(self.hidden_dim)(d1)
        d1 = nn.swish(d1)
        item_logits = nn.Dense(self.corpus_size, name="item_logits")(d1)
        
        # --- Decoder Head 2: Rating Prediction ---
        d2 = nn.Dense(self.hidden_dim // 2)(z)
        d2 = nn.swish(d2)
        d2 = nn.Dense(self.hidden_dim)(d2)
        d2 = nn.swish(d2)
        rating_pred = nn.Dense(self.corpus_size, name="rating_pred")(d2)
        
        # Learnable uncertainty weights for multi-task loss
        log_var_presence = self.param('log_var_presence', nn.initializers.zeros, (1,))
        log_var_rating = self.param('log_var_rating', nn.initializers.zeros, (1,))
        
        return item_logits, rating_pred, log_var_presence, log_var_rating

## 4. Training Loop
We train the model using Multinomial Log-Likelihood for the presence head and masked Huber loss for the rating head. Input profiles are dynamically corrupted (dropout) during training.

In [10]:
class TrainState(train_state.TrainState):
    key: jax.Array

def create_train_state(rng, learning_rate):
    model = Recommender()
    dummy_input = jnp.ones((1, CORPUS_SIZE * 2))
    params = model.init({"params": rng, "noise": rng}, dummy_input)["params"]
    tx = optax.adam(learning_rate)
    return TrainState.create(apply_fn=model.apply, params=params, tx=tx, key=rng)

@jax.jit
def train_step(state, batch, rated_mask):
    presence = batch[:, :CORPUS_SIZE]
    ratings_z = batch[:, CORPUS_SIZE:]
    
    dropout_rng, vae_rng = random.split(state.key)
    
    # Dynamic dropout to act as denoising autoencoder
    keep = random.bernoulli(dropout_rng, p=(1.0 - DROPOUT_RATE), shape=presence.shape)
    corrupted_presence = presence * keep
    corrupted_ratings = ratings_z * keep
    x_in = jnp.concatenate([corrupted_presence, corrupted_ratings], axis=1)
    
    def loss_fn(params):
        item_logits, rating_pred, log_var_presence, log_var_rating = state.apply_fn(
            {"params": params}, x_in, training=True, rngs={"noise": vae_rng}
        )
        
        # Presence Multinomial Loss
        log_probs = jax.nn.log_softmax(item_logits, axis=1)
        per_user_counts = jnp.maximum(jnp.sum(presence, axis=1), 1.0)
        presence_loss = jnp.mean(-jnp.sum(presence * log_probs, axis=1) / per_user_counts)
        
        # Rating Huber Loss (masked to actually rated items)
        err = rating_pred - ratings_z
        per_entry_loss = optax.huber_loss(err, delta=1.0)
        denom_r = jnp.maximum(jnp.sum(rated_mask, axis=1), 1.0)
        rating_loss = jnp.mean(jnp.sum(rated_mask * per_entry_loss, axis=1) / denom_r)
        
        # Uncertainty Weighting
        precision_p = jnp.exp(-log_var_presence)
        precision_r = jnp.exp(-log_var_rating)
        weighted_loss = (precision_p * presence_loss + log_var_presence) + \
                        (precision_r * rating_loss + log_var_rating)
        
        return jnp.mean(weighted_loss), (presence_loss, rating_loss)
        
    (loss, (recon_p, recon_r)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    state = state.replace(key=dropout_rng)
    return state, loss, recon_p, recon_r

# Data Generator (Memory Efficient)
def data_generator(X_sparse, M_sparse, batch_size):
    num_samples = X_sparse.shape[0]
    indices = np.arange(num_samples)
    np.random.shuffle(indices)
    for i in range(0, num_samples, batch_size):
        batch_idx = indices[i:i+batch_size]
        # Convert only the batch to dense arrays to save RAM
        yield X_sparse[batch_idx].toarray(), M_sparse[batch_idx].toarray()

# Initialization & Training
rng = random.PRNGKey(42)
state = create_train_state(rng, LEARNING_RATE)

print("Starting Training...")
for epoch in range(EPOCHS):
    gen = data_generator(X_train_sparse, M_train_sparse, BATCH_SIZE)
    epoch_losses = []
    num_batches = int(np.ceil(X_train_sparse.shape[0] / BATCH_SIZE))
    
    # Progress bar with stats
    pbar = tqdm(gen, total=num_batches, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch_x, batch_m in pbar:
        state, loss, p_loss, r_loss = train_step(state, jnp.array(batch_x), jnp.array(batch_m))
        epoch_losses.append(loss)
        pbar.set_postfix(loss=f"{np.mean(epoch_losses):.4f}", p_loss=f"{float(p_loss):.4f}", r_loss=f"{float(r_loss):.4f}")
        
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {np.mean(epoch_losses):.4f}")

# Save model weights for backend use
from flax import serialization
with open('/kaggle/working/model_weights.msgpack', 'wb') as f:
    f.write(serialization.to_bytes(state.params))
    
print("Saved model weights to /kaggle/working/model_weights.msgpack")

Starting Training...


Epoch 1/10:   0%|          | 0/573 [00:00<?, ?it/s]

Epoch 1/10 - Loss: 6.7422


Epoch 2/10:   0%|          | 0/573 [00:00<?, ?it/s]

Epoch 2/10 - Loss: 5.5783


...

Epoch 8/10 - Loss: 2.5908


Epoch 9/10:   0%|          | 0/573 [00:00<?, ?it/s]

Epoch 9/10 - Loss: 2.3441


Epoch 10/10:   0%|          | 0/573 [00:00<?, ?it/s]

Epoch 10/10 - Loss: 2.1397
Saved model weights to /kaggle/working/model_weights.msgpack


## 5. Generating Recommendations
Once trained, we can perform inference. We combine the predicted item probability and rating into a final recommendation score.

In [12]:
def get_recommendations(state, user_idx, top_k=20, logit_weight=0.3):
    user_input = X_train_sparse[user_idx:user_idx+1].toarray()
    item_logits, rating_pred, _, _ = state.apply_fn(
        {"params": state.params}, jnp.array(user_input), training=False
    )
    
    logits_1d = item_logits[0]
    preds_1d = rating_pred[0]
    presence_mask = user_input[0, :CORPUS_SIZE]
    
    # Z-Score the logits and ratings for linear mixing
    norm_logits = (logits_1d - jnp.mean(logits_1d)) / (jnp.std(logits_1d) + 1e-6)
    norm_ratings = (preds_1d - jnp.mean(preds_1d)) / (jnp.std(preds_1d) + 1e-6)
    
    combined_score = (logit_weight * norm_logits) + ((1.0 - logit_weight) * norm_ratings)
    
    # Mask out items already in the user's profile
    masked_score = jnp.where(presence_mask > 0, -jnp.inf, combined_score)
    
    # Get top K
    top_indices = jnp.argsort(masked_score)[-top_k:][::-1]
    top_scores = masked_score[top_indices]
    
    # Map corpus indices back to anime IDs
    anime_ids = [corpus_ids[i] for i in top_indices]
    return anime_ids, top_scores

print("Sample Recommendations for User 0:")
recs, scores = get_recommendations(state, user_idx=10)
for rank, (aid, score) in enumerate(zip(recs, scores), 1):
    print(f"{rank}. Anime ID: {aid} (Score: {score:.3f})")

Sample Recommendations for User 0:
1. Anime ID: 3002 (Score: 2.263)
2. Anime ID: 35180 (Score: 2.250)
3. Anime ID: 2921 (Score: 2.226)
4. Anime ID: 35247 (Score: 2.159)
5. Anime ID: 6594 (Score: 2.155)
6. Anime ID: 38524 (Score: 2.136)
7. Anime ID: 49387 (Score: 2.073)
8. Anime ID: 7311 (Score: 2.014)
9. Anime ID: 205 (Score: 2.013)
10. Anime ID: 25537 (Score: 1.985)
11. Anime ID: 3786 (Score: 1.963)
12. Anime ID: 37991 (Score: 1.955)
13. Anime ID: 170 (Score: 1.953)
14. Anime ID: 6114 (Score: 1.952)
15. Anime ID: 2001 (Score: 1.949)
16. Anime ID: 37521 (Score: 1.947)
17. Anime ID: 10271 (Score: 1.947)
18. Anime ID: 33050 (Score: 1.931)
19. Anime ID: 33049 (Score: 1.930)
20. Anime ID: 457 (Score: 1.910)
